In [1]:
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm

from diffusers import StableDiffusionPipeline
from torchvision import transforms

from torchmetrics.image.fid import FrechetInceptionDistance
from prdc import compute_prdc
import random


/home/.conda/envs/lora_sd/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None
).to("cuda")

pipe.load_lora_weights(
    "lora_output/new_experiment",
    weight_name="last.safetensors"
)

pipe.enable_attention_slicing()
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()


Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00,  6.40it/s]
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


In [3]:
import open_clip

device = "cuda" if torch.cuda.is_available() else "cpu"

model, preprocess, tokenizer = open_clip.create_model_and_transforms(
    "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
)

model = model.to(device)
model.eval()

/home/.conda/envs/lora_sd/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


CustomTextCLIP(
  (visual): TimmModel(
    (trunk): VisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
        (norm): Identity()
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (patch_drop): Identity()
      (norm_pre): Identity()
      (blocks): Sequential(
        (0): Block(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (attn): Attention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (proj): Linear(in_features=768, out_features=768, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): Identity()
          (drop_path1): Identity()
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
          

In [4]:
def extract_biomedclip_features(images):

    feats = []

    with torch.no_grad():
        for img in images:

            x = preprocess(img).unsqueeze(0).to(device)

            f = model.encode_image(x)

            f = f / f.norm(dim=-1, keepdim=True)

            feats.append(f.cpu().numpy())

    return np.vstack(feats)

In [5]:
from scipy import linalg

def compute_fid_from_features(real_feats, fake_feats):

    mu1 = real_feats.mean(axis=0)
    mu2 = fake_feats.mean(axis=0)

    sigma1 = np.cov(real_feats, rowvar=False)
    sigma2 = np.cov(fake_feats, rowvar=False)

    diff = mu1 - mu2

    covmean = linalg.sqrtm(sigma1 @ sigma2)

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff @ diff + np.trace(sigma1 + sigma2 - 2 * covmean)

    return float(fid)

In [6]:
def compute_clip_similarity(fake_imgs, text):
    # Используем токенизатор из open_clip
    text_tokens = tokenizer([text], return_tensors="pt").to(device)

    with torch.no_grad():
        text_feat = model.encode_text(text_tokens)
        text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)

    sims = []

    for img in fake_imgs:
        x = preprocess(img).unsqueeze(0).to(device)

        with torch.no_grad():
            img_feat = model.encode_image(x)
            img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)

        sim = (img_feat @ text_feat.T).item()
        sims.append(sim)

    return np.mean(sims)

In [7]:
def compute_prd(real_imgs, fake_imgs):
    real_feats = extract_biomedclip_features(real_imgs)
    fake_feats = extract_biomedclip_features(fake_imgs)

    return compute_prdc(
        real_features=real_feats,
        fake_features=fake_feats,
        nearest_k=5
    )


In [8]:
METADATA = Path("/home/project/datasets/BCN_new/metadata.csv")
IMG_DIR = Path("/home/project/datasets/BCN_new")

df = pd.read_csv(METADATA)

In [9]:
def generate_images_batched(pipe, cls, n, batch_size=4):
    """
    Генерация изображений батчами через StableDiffusionPipeline с LoRA.

    pipe : StableDiffusionPipeline
    cls : str, класс заболевания
    n : int, количество изображений
    batch_size : int, размер батча
    """
    prompt = f"<lesion> dermoscopy image of {PROMPT_MAP[cls]}"
    negative = "clinical photo, ruler, text, watermark, low quality"

    images = []

    for i in range(0, n, batch_size):
        current_batch_size = min(batch_size, n - i)
        with torch.no_grad(), torch.cuda.amp.autocast():
            batch = pipe(
                [prompt] * current_batch_size,  # список одинаковых промптов
                negative_prompt=[negative] * current_batch_size,
                num_images_per_prompt=1,
                guidance_scale=7.0,
                num_inference_steps=40
            ).images

        images.extend(batch)

    return images

In [10]:
def load_real_images(df, image_dir, cls, n=5):

    df_cls = df[df["class"] == cls]

    if len(df_cls) == 0:
        return []

    df_cls = df_cls.sample(n=min(n, len(df_cls)), random_state=42)

    images = []

    for isic_id in df_cls["isic_id"]:
        img_path = Path(image_dir) / f"{isic_id}.jpg"

        if img_path.exists():
            images.append(Image.open(img_path).convert("RGB"))

    print(f"Loaded images → real: {len(images)}")

    return images

In [ ]:
MIN_REAL = 100
N_GEN = 100

results = []

DIAGNOSIS_MAP = {
    "Solar or actinic keratosis": "AK",
    "Basal cell carcinoma": "BCC",
    "Seborrheic keratosis": "BKL",
    "Solar lentigo": "BKL",
    "Dermatofibroma": "DF",
    "Melanoma metastasis": "MEL",
    "Melanoma, NOS": "MEL",
    "Nevus": "NV",
    "Squamous cell carcinoma, NOS": "SCC",
}

PROMPT_MAP = {
    "AK": "Actinic keratosis",
    "BCC": "Basal cell carcinoma",
    "BKL": "Benign keratosis lesion",
    "DF": "Dermatofibroma",
    "MEL": "Melanoma",
    "NV": "Nevus",
    "SCC": "Squamous cell carcinoma",
}

df = pd.read_csv(METADATA)
df["class"] = df["diagnosis_3"].map(DIAGNOSIS_MAP)

# удаляем все строки без класса
df = df.dropna(subset=["class"])
classes = sorted(df["class"].unique())

print(f"Found {len(classes)} classes: {classes}")

for cls in classes:

    print(f"\n=== {cls} ===")

    real_imgs = load_real_images(df, IMG_DIR, cls, N_GEN)

    if len(real_imgs) < 2:
        continue

    fake_imgs = generate_images_batched(pipe, cls, N_GEN, batch_size=8)

    real_feats = extract_biomedclip_features(real_imgs)
    fake_feats = extract_biomedclip_features(fake_imgs)

    fid = compute_fid_from_features(real_feats, fake_feats)
    prd = compute_prd(real_imgs, fake_imgs)

    clip_sim = compute_clip_similarity(
        fake_imgs,
        f"dermoscopy image of {PROMPT_MAP[cls]}"
    )

   
    results.append({
        "class": cls,
        "n_real": len(real_imgs),
        "fid": fid,
        "precision": prd["precision"],
        "recall": prd["recall"],
        "clip": clip_sim,
    })

    print(f"FID: {fid:.2f}")
    print(f"Precision: {prd['precision']:.3f}")
    print(f"Recall: {prd['recall']:.3f}")
    print(f"CLIP: {clip_sim:.3f}")

    del fake_imgs
    torch.cuda.empty_cache()


Found 7 classes: ['AK', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'SCC']

=== AK ===
Loaded images → real: 100


  0%|          | 0/40 [00:00<?, ?it/s]/home/.conda/envs/lora_sd/lib/python3.10/site-packages/torch/nn/modules/conv.py:456: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv2d(input, weight, bias, self.stride,
 68%|██████▊   | 27/40 [01:23<00:38,  2.97s/it]